# Eval Workflow

Notes for this chapter:

- 

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [19]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stream=False, output_config=None, output_format=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if output_config:
        params["output_config"] = output_config

    if output_format:
        params["output_format"] = output_format

    if stream:
        with client.messages.stream(**params) as response:
            for text in response.text_stream:
                print(text, end="", flush=True)
            message = response.get_final_message()
    else:
        if output_format:
            message = client.messages.parse(**params)
        else:
            message = client.messages.create(**params)

    # Responses can start with a thinking block, so find the text block
    return next(block.text for block in message.content if block.type == "text")

In [3]:
messages = []

add_user_message(messages, "Hello!")
answer = chat(messages)
add_assistant_message(messages, answer)

answer

'Hello! How can I help you today?'

## Exercises

# Building an eval pipeline for generating output files

Our prompt needs to assist users in writing three specific types of output for AWS use cases:

- Python code
- JSON configuration files
- Regular expressions

In [8]:
import json
from pathlib import Path

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    output_config = {
          "format": {
              "type": "json_schema",
              "schema": {
                  "type": "object",
                  "properties": {
                      "tasks": {
                          "type": "array",
                          "items": {
                              "type": "object",
                              "properties": {
                                  "task": {"type": "string"},
                              },
                              "required": ["task"],
                              "additionalProperties": False,
                          },
                      },
                  },
                  "required": ["tasks"],
                  "additionalProperties": False,
              },
          }
      }

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, output_config=output_config)
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

path = Path("datasets/eval.json")
path.parent.mkdir(parents=True, exist_ok=True)

with open(path, 'w') as f:
    json.dump(dataset, f, indent=2)


{'tasks': [{'task': 'Write a Python function that takes an S3 bucket name and a prefix, uses boto3 to list all objects under that prefix (handling pagination), and returns the total size in bytes of all those objects.'}, {'task': "Write a JSON IAM policy document that grants read-only access (s3:GetObject and s3:ListBucket) to a bucket named 'my-app-logs' and all objects within it, with no other permissions."}, {'task': "Write a regular expression that matches a valid AWS EC2 instance ID, which is the prefix 'i-' followed by either 8 or 17 lowercase hexadecimal characters, and that matches only if the entire string is an instance ID."}]}


In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset["tasks"]:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

with open("datasets/eval.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a function that does this using boto3's built-in paginator:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\n\ndef get_total_size_bytes(bucket: str, prefix: str = \"\", client=None) -> int:\n    \"\"\"\n    Return the total size in bytes of all objects in an S3 bucket under a prefix.\n\n    Args:\n        bucket: Name of the S3 bucket.\n        prefix: Key prefix to filter objects (e.g. \"logs/2024/\"). Empty = whole bucket.\n        client: Optional boto3 S3 client (useful for testing or custom config).\n\n    Returns:\n        Total size of all matching objects, in bytes.\n\n    Raises:\n        botocore.exceptions.ClientError: on AWS errors (e.g. NoSuchBucket, AccessDenied).\n    \"\"\"\n    s3 = client or boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n\n    total_size = 0\n    try:\n        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):\n            # \"Contents\" is absent when a pag

In [21]:
from pydantic import BaseModel

class Grading(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int


def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    text = chat(messages, output_format=Grading)
    print(text)
    return json.loads(text)

sum = 0

for result in results:
    grading = grade_by_model(result["test_case"]["task"], result["output"])
    sum += grading["score"]

print(f"Average is {sum/len(results)}")


{"strengths":["Correctly uses boto3's list_objects_v2 paginator so pagination is handled reliably, and uses page.get('Contents', []) to handle empty pages.","Clean, typed signature with docstring and an injectable client parameter, which makes the function easy to test.","Helpful notes on prefix semantics, versioned buckets, permissions, and performance for large prefixes."],"weaknesses":["The docstring says it raises ClientError, but the code catches ClientError and re-raises RuntimeError, so the docs and behavior contradict each other.","Wrapping ClientError in RuntimeError discards the specific exception type, making it harder for callers to handle cases like NoSuchBucket or AccessDenied programmatically.","The note that console-created folder placeholders have size 0 is fine, but the answer doesn't mention that the paginator may not cover edge cases like Requester Pays buckets or customizing the signature (minor)."],"reasoning":"The core logic is correct, idiomatic, and meets the t